Dimensões da Camada Ouro

Construir as dimensões analíticas da camada Ouro a partir dos dados
tratados e harmonizados na camada Prata.

Dimensões previstas:

- `dim_fonte`
- `dim_filme`
- `dim_data`

A camada Ouro será utilizada posteriormente para construção da tabela
fato de avaliações e para as análises de negócio.

### Princípios adotados

- utilizar a camada Prata como fonte
- não modificar as tabelas da camada Prata
- preservar a rastreabilidade dos dados
- utilizar somente campos existentes e previamente validados
- evitar transformações sem regra de negócio definida
- produzir tabelas Delta na camada Ouro

In [0]:
# ============================================================
# Configuração, leitura e validação do contrato de entrada
# ============================================================

from pyspark.sql import functions as F

TABELA_PRATA = "mvp3_eng_dados.prata.avaliacoes_harmonizadas"

TABELA_DIM_FONTE = "mvp3_eng_dados.ouro.dim_fonte"
TABELA_DIM_FILME = "mvp3_eng_dados.ouro.dim_filme"
TABELA_DIM_DATA = "mvp3_eng_dados.ouro.dim_data"

df_prata = spark.table(TABELA_PRATA)

colunas_necessarias = {
    "id_avaliacao_global",
    "fonte",
    "id_filme",
    "titulo_filme",
    "link_filme",
    "data_avaliacao"
}

colunas_existentes = set(df_prata.columns)
colunas_ausentes = colunas_necessarias - colunas_existentes

print(f"Tabela de entrada: {TABELA_PRATA}")
print(f"Quantidade de colunas: {len(df_prata.columns)}")
print(f"Colunas necessárias: {sorted(colunas_necessarias)}")
print(f"Colunas ausentes: {sorted(colunas_ausentes)}")

assert len(colunas_ausentes) == 0, (
    f"Contrato de entrada inválido. "
    f"Colunas ausentes: {sorted(colunas_ausentes)}"
)

print("\n✓ Contrato de entrada validado.")

Tabela de entrada: mvp3_eng_dados.prata.avaliacoes_harmonizadas
Quantidade de colunas: 16
Colunas necessárias: ['data_avaliacao', 'fonte', 'id_avaliacao_global', 'id_filme', 'link_filme', 'titulo_filme']
Colunas ausentes: []

✓ Contrato de entrada validado.


In [0]:
# ============================================================
# Perfil dos campos utilizados pelas dimensões
# ============================================================

total_prata = df_prata.count()

df_perfil_dimensoes = df_prata.agg(
    F.countDistinct("fonte").alias("fontes_distintas"),
    F.countDistinct("id_filme").alias("ids_filme_distintos"),
    F.countDistinct("data_avaliacao").alias("datas_distintas"),

    F.sum(
        F.when(F.col("fonte").isNull(), 1).otherwise(0)
    ).alias("fonte_nula"),

    F.sum(
        F.when(F.col("id_filme").isNull(), 1).otherwise(0)
    ).alias("id_filme_nulo"),

    F.sum(
        F.when(F.col("data_avaliacao").isNull(), 1).otherwise(0)
    ).alias("data_avaliacao_nula")
)

print(f"Registros recebidos da Prata: {total_prata:,}")

display(df_perfil_dimensoes)

Registros recebidos da Prata: 1,060,128


fontes_distintas,ids_filme_distintos,datas_distintas,fonte_nula,id_filme_nulo,data_avaliacao_nula
2,17706,8015,0,49712,49582


In [0]:
# ============================================================
# Construção da dimensão fonte
# ============================================================

df_dim_fonte = (
    df_prata
    .select("fonte")
    .filter(F.col("fonte").isNotNull())
    .distinct()
    .withColumn(
        "id_fonte",
        F.when(F.col("fonte") == "IMDb", F.lit(1))
         .when(F.col("fonte") == "Rotten Tomatoes", F.lit(2))
         .otherwise(F.lit(99))
    )
    .select(
        "id_fonte",
        "fonte"
    )
    .orderBy("id_fonte")
)

display(df_dim_fonte)

id_fonte,fonte
1,IMDb
2,Rotten Tomatoes


In [0]:
# ============================================================
# Validação e persistência da dimensão fonte
# ============================================================

fontes_inesperadas = (
    df_dim_fonte
    .filter(F.col("id_fonte") == 99)
    .count()
)

duplicidades_fonte = (
    df_dim_fonte.count()
    - df_dim_fonte.select("fonte").distinct().count()
)

assert fontes_inesperadas == 0, (
    "Foi encontrada fonte diferente de IMDb e Rotten Tomatoes."
)

assert duplicidades_fonte == 0, (
    "Foram encontradas fontes duplicadas na dimensão."
)

(
    df_dim_fonte
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_DIM_FONTE)
)

print(f"✓ {TABELA_DIM_FONTE} gravada com sucesso.")

✓ mvp3_eng_dados.ouro.dim_fonte gravada com sucesso.


In [0]:
# ============================================================
# Construção da dimensão filme
# ============================================================

df_dim_filme = (
    df_prata
    .filter(F.col("id_filme").isNotNull())
    .select(
        "id_filme",
        "titulo_filme",
        "link_filme"
    )
    .dropDuplicates()
)

print(
    f"Registros candidatos à dimensão filme: "
    f"{df_dim_filme.count():,}"
)

display(
    df_dim_filme
    .orderBy("titulo_filme")
    .limit(20)
)

Registros candidatos à dimensão filme: 17,706


id_filme,titulo_filme,link_filme
87c3f32dbdeba1a3c85d531d37504439f5d169cdb2c9064e5626164bb10a85fb,!Women Art Revolution,m/women_art_revolution_a_secret_history
3961b93f842187b6cda20952ad3ef0f72548322a381f7d1963d151490e4e8ed3,#Horror,m/horror_2015
497b59c39ea2ba0504e6151d64f13cba6690f715cbc92849f7abdb44d48b73df,#Unfit: The Psychology of Donald Trump,m/unfit_the_psychology_of_donald_trump
088df0009b9581dc3b1618f335867a1d87c48f0c65d2eb36cc2825f8223cb97c,$ (Dollars) (The Heist),m/dollars
0479fff2d3592e2831216d88e40366cc416a9c28cf1a6486b7572b93d6accaaa,$9.99,m/999_2008
778130a3b934eb774d0a54fadb6279a8da63807c51205fe0254cb827c0e42370,$ellebrity,m/ellebrity
6fe01de5748a96ba12887c3989704efaf6a57cd95fea2759a3268aa0083d8850,'71,m/71_2015
c712e417812d0c77c9f245c4da403fcfffc88f1251f9ddaf84ac0cee652efbb6,'Breaker' Morant,m/breaker_morant
ff4241af7e53a05bf67a45aec4559a9c2dd3f6032bb4180bfbffb4c376c6cb21,'R Xmas,m/r-xmas
ff16fc789d328daf5e785fa621a26e4876dbc4d661333c1e960a7a344583e3ee,'Round Midnight,m/round_midnight


In [0]:
# ============================================================
# Validação da unicidade de id_filme
# ============================================================

df_conflitos_filme = (
    df_dim_filme
    .groupBy("id_filme")
    .agg(
        F.count("*").alias("quantidade_variacoes")
    )
    .filter(
        F.col("quantidade_variacoes") > 1
    )
)

qtd_conflitos_filme = df_conflitos_filme.count()

print(
    f"IDs de filme associados a mais de uma combinação "
    f"de atributos: {qtd_conflitos_filme:,}"
)

if qtd_conflitos_filme > 0:
    display(df_conflitos_filme.limit(20))

assert qtd_conflitos_filme == 0, (
    "Existem conflitos de atributos para o mesmo id_filme."
)

print("✓ Unicidade semântica de id_filme validada.")

IDs de filme associados a mais de uma combinação de atributos: 0
✓ Unicidade semântica de id_filme validada.


In [0]:
# ============================================================
# Persistência da dimensão filme
# ============================================================

(
    df_dim_filme
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_DIM_FILME)
)

total_dim_filme = spark.table(TABELA_DIM_FILME).count()

print(f"Filmes persistidos: {total_dim_filme:,}")
print(f"✓ {TABELA_DIM_FILME} gravada com sucesso.")

Filmes persistidos: 17,706
✓ mvp3_eng_dados.ouro.dim_filme gravada com sucesso.


In [0]:
# ============================================================
# Construção da dimensão data
# ============================================================

df_dim_data = (
    df_prata
    .select("data_avaliacao")
    .filter(F.col("data_avaliacao").isNotNull())
    .distinct()

    .withColumn(
        "id_data",
        F.date_format(
            F.col("data_avaliacao"),
            "yyyyMMdd"
        ).cast("int")
    )

    .withColumn(
        "ano",
        F.year("data_avaliacao")
    )

    .withColumn(
        "mes",
        F.month("data_avaliacao")
    )

    .withColumn(
        "dia",
        F.dayofmonth("data_avaliacao")
    )

    .withColumn(
        "trimestre",
        F.quarter("data_avaliacao")
    )

    .select(
        "id_data",
        "data_avaliacao",
        "ano",
        "trimestre",
        "mes",
        "dia"
    )

    .orderBy("data_avaliacao")
)

display(df_dim_data.limit(20))

id_data,data_avaliacao,ano,trimestre,mes,dia
18000101,1800-01-01,1800,1,1,1
19230101,1923-01-01,1923,1,1,1
19270812,1927-08-12,1927,3,8,12
19270923,1927-09-23,1927,3,9,23
19280421,1928-04-21,1928,2,4,21
19290201,1929-02-01,1929,1,2,1
19310126,1931-01-26,1931,1,1,26
19320101,1932-01-01,1932,1,1,1
19330101,1933-01-01,1933,1,1,1
19340222,1934-02-22,1934,1,2,22


In [0]:
# ============================================================
# Validação e persistência da dimensão data
# ============================================================

total_datas = df_dim_data.count()

ids_data_distintos = (
    df_dim_data
    .select("id_data")
    .distinct()
    .count()
)

assert total_datas == ids_data_distintos, (
    "Foram encontrados IDs duplicados na dimensão data."
)

assert (
    df_dim_data
    .filter(F.col("id_data").isNull())
    .count()
) == 0, (
    "Foram encontrados IDs nulos na dimensão data."
)

(
    df_dim_data
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_DIM_DATA)
)

print(f"Datas persistidas: {total_datas:,}")
print(f"✓ {TABELA_DIM_DATA} gravada com sucesso.")

Datas persistidas: 8,015
✓ mvp3_eng_dados.ouro.dim_data gravada com sucesso.


In [0]:
# ============================================================
# Validação das dimensões persistidas
# ============================================================

df_fonte_validacao = spark.table(TABELA_DIM_FONTE)
df_filme_validacao = spark.table(TABELA_DIM_FILME)
df_data_validacao = spark.table(TABELA_DIM_DATA)

resultado_validacao = [
    (
        "dim_fonte",
        df_fonte_validacao.count(),
        df_fonte_validacao.select("id_fonte").distinct().count()
    ),
    (
        "dim_filme",
        df_filme_validacao.count(),
        df_filme_validacao.select("id_filme").distinct().count()
    ),
    (
        "dim_data",
        df_data_validacao.count(),
        df_data_validacao.select("id_data").distinct().count()
    )
]

df_resultado_validacao = spark.createDataFrame(
    resultado_validacao,
    [
        "dimensao",
        "registros",
        "ids_distintos"
    ]
)

display(df_resultado_validacao)

dimensao,registros,ids_distintos
dim_fonte,2,2
dim_filme,17706,17706
dim_data,8015,8015


In [0]:
# ============================================================
# Validações críticas finais
# ============================================================

problemas_unicidade = (
    df_resultado_validacao
    .filter(
        F.col("registros") != F.col("ids_distintos")
    )
    .count()
)

assert problemas_unicidade == 0, (
    "Uma ou mais dimensões possuem chaves duplicadas."
)

assert df_fonte_validacao.count() > 0
assert df_filme_validacao.count() > 0
assert df_data_validacao.count() > 0

print("✓ dim_fonte validada")
print("✓ dim_filme validada")
print("✓ dim_data validada")
print("✓ unicidade das chaves validada")
print("✓ persistência Delta validada")

✓ dim_fonte validada
✓ dim_filme validada
✓ dim_data validada
✓ unicidade das chaves validada
✓ persistência Delta validada


In [0]:
# ============================================================
# Encerramento
# ============================================================

print("=" * 65)
print("09_DIMENSOES_OURO — CONCLUÍDO")
print("=" * 65)

print("\nDimensões criadas:")
print(f"✓ {TABELA_DIM_FONTE}")
print(f"✓ {TABELA_DIM_FILME}")
print(f"✓ {TABELA_DIM_DATA}")

print("\nPrincípios preservados:")
print("✓ camada Prata não modificada")
print("✓ identificadores existentes reutilizados")
print("✓ novas chaves criadas somente quando necessárias")
print("✓ dimensões persistidas em Delta")
print("✓ unicidade das chaves validada")

print(
    "\nPróxima etapa: "
    "10_fato_avaliacao_ouro"
)

09_DIMENSOES_OURO — CONCLUÍDO

Dimensões criadas:
✓ mvp3_eng_dados.ouro.dim_fonte
✓ mvp3_eng_dados.ouro.dim_filme
✓ mvp3_eng_dados.ouro.dim_data

Princípios preservados:
✓ camada Prata não modificada
✓ identificadores existentes reutilizados
✓ novas chaves criadas somente quando necessárias
✓ dimensões persistidas em Delta
✓ unicidade das chaves validada

Próxima etapa: 10_fato_avaliacao_ouro
